<div style="display: flex; background-color: #5B6C9F;" >
<h1 style="margin: auto; padding: 30px; color: white; ">Détectez des faux billets - Notebook 2 - Application du modèle</h1>
</div>

In [1]:
# Importation des librairies
import pandas as pd
import joblib
import numpy as np

In [ ]:
# Chargement du modèle
modele = joblib.load("../model/counterfeit_detection_model.joblib")

In [ ]:
# Chargement des nouvelles données
df_nouveaux_billets = pd.read_csv("../data/billets_production.csv", sep=",")

In [4]:
# Sélection des variables de taille des billets
taille_modele = [
    "diagonal",
    "height_left",
    "height_right",
    "margin_low",
    "margin_up",
    "length",
]

In [5]:
# Creation d'un dataframe avec seulement les mesures
df_taille_billets = df_nouveaux_billets[taille_modele]

In [6]:
# Mise en place de la prédiction sur les nouvelles données
predictions = modele.predict(df_taille_billets)

In [7]:
# Confiance du modèle derrière sa prise de décision
confiance = modele.predict_proba(df_taille_billets)[:, 1]

In [8]:
# Création d'une copie pour ne pas modifier les données d'origine
resultats = df_nouveaux_billets[["id"]].copy()

In [9]:
# Traduction des prédictions en tableau
resultats["Prédictions"] = ["Vrai billet" if p else "Faux billet" for p in predictions]
resultats["Probabilités d'être vrai"] = confiance.round(3)

In [10]:
# Mise en place d'un seuil d'alerte des billets incertains pour vérification humaine
ecart = np.abs(confiance - 0.5) * 2
seuil = 0.30
resultats["Inspection humaine"] = np.where(ecart < seuil, "À vérifier", "")

In [11]:
# Affichage des résultats
resultats

,id,Prédictions,Probabilités d'être vrai,Inspection humaine
0,B_1,Vrai billet,0.995,
1,B_2,Faux billet,0.003,
2,B_3,Vrai billet,0.999,
3,B_4,Faux billet,0.000,
4,B_5,Faux billet,0.011,


In [12]:
# Bilan comptable des vrais et faux billets
bilan = resultats["Prédictions"].value_counts()
bilan

Prédictions
Faux billet    3
Vrai billet    2
Name: count, dtype: int64